### IMPORTS DE LIBRERIAS Y CARGA DE DATOS

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/server_logs.csv')

### ANALISIS DE DATOS PREVIO

In [3]:
# Cambiamos a tipo datetime sino pandas lo trataría como string
df['timestamp_event'] = pd.to_datetime(df['timestamp_event'])

rango = pd.DataFrame({
    'rango-fechas': [df['timestamp_event'].min(), df['timestamp_event'].max()]
}, index=['Desde', 'Hasta'])

display(rango)

# Columna booleana donde se almacenan los eventos graves
df['is_bad'] = (
    (df['severity'].isin(['ERROR', 'CRITICAL'])) |
    (df['status_code'] >= 500)
)

resumen_general = pd.DataFrame({
    'cantidad-logs': [len(df), df['is_bad'].sum(), round(df['is_bad'].mean() * 100, 2)]
}, index=['Total eventos', 'Eventos malos', 'Bad rate global (%)'])

display(resumen_general)

display(df['severity'].value_counts().to_frame('cantidad-severidad'))

# Captura cantidad de valores únicos dentro de la columna de servicio
conteo_servicios = df['service_name'].value_counts().to_frame('cantidad-servicio')

display(conteo_servicios.head(6))

display(df['message'].value_counts().head().to_frame('cantidad-mensaje-general'))

display(df[df['is_bad']]['message'].value_counts().head().to_frame('cantidad-mensaje-malos'))

,rango-fechas
Desde,2026-01-10 00:02:39.029160+00:00
Hasta,2026-01-12 23:59:23.187914+00:00


,cantidad-logs
Total eventos,5795.00
Eventos malos,895.00
Bad rate global (%),15.44


,cantidad-severidad
severity,
INFO,3542
WARN,1358
ERROR,775
CRITICAL,120


,cantidad-servicio
service_name,
api-gateway,1509
orders-service,1057
inventory-service,964
payment-service,842
auth-service,778
notification-service,645


,cantidad-mensaje-general
message,
Health check OK,1196
Background job completed,1185
Request completed,1161
Order creation failed - inventory lock timeout,197
Rate limit nearing threshold,193


,cantidad-mensaje-malos
message,
Order creation failed - inventory lock timeout,197
Payment gateway unavailable,103
Database deadlock detected,99
Checkout failed - upstream payment timeout,88
Possible credential stuffing detected,69


### DETECCION DEL MOMENTO CRITICO (VENTANAS DE 5 MINUTOS) 

In [21]:
# Crear bins de 5 minutos
df['window'] = df['timestamp_event'].dt.floor('5min')

# Agrupar por ventana y calcular métricas
resumen = df.groupby('window').agg(
    total_events = ('is_bad', 'count'),
    bad_events   = ('is_bad', 'sum')
).reset_index()

# Calcular bad_rate por ventana
resumen['bad_rate'] = resumen['bad_events'] / resumen['total_events']

# Filtro obligatorio: ventanas con al menos 20 eventos
resumen_filtrado = resumen[resumen['total_events'] >= 20].copy()

# Top 5 por bad_rate
top5 = resumen_filtrado.sort_values('bad_rate', ascending=False).head(5)
top5['bad_rate'] = top5['bad_rate'].round(4)

display(top5)

,window,total_events,bad_events,bad_rate
134,2026-01-10 11:10:00+00:00,189,110,0.5820
135,2026-01-10 11:15:00+00:00,228,129,0.5658
136,2026-01-10 11:20:00+00:00,111,59,0.5315
462,2026-01-11 14:35:00+00:00,255,117,0.4588
461,2026-01-11 14:30:00+00:00,156,68,0.4359


### IDENTIFICAR EL MOMENTO CRITICO

In [22]:
# El momento crítico es el top 1 del resumen filtrado
momento_critico = resumen_filtrado.sort_values('bad_rate', ascending=False).iloc[0]
ventana_critica = momento_critico['window']

resumen_critico = pd.DataFrame({
    'valor': [
        ventana_critica,
        momento_critico['total_events'],
        momento_critico['bad_events'],
        round(momento_critico['bad_rate'] * 100, 2)
    ]
}, index=['Ventana', 'Total eventos', 'Bad events', 'Bad rate (%)'])

print("MOMENTO CRÍTICO DETECTADO:")
display(resumen_critico)

MOMENTO CRÍTICO DETECTADO:


,valor
Ventana,2026-01-10 11:10:00+00:00
Total eventos,189
Bad events,110
Bad rate (%),58.2


### FILTRAR LOS EVENTOS DEL MOMENTO CRITICO

In [24]:
# Filtramos solo los eventos que cayeron en la ventana crítica
df_critico = df[df['window'] == ventana_critica].copy()

print("Eventos en la ventana crítica:", len(df_critico))
print("Bad events en la ventana crítica:", df_critico['is_bad'].sum())

Eventos en la ventana crítica: 189
Bad events en la ventana crítica: 110


### BAD EVENTS POR SERVICIO

In [25]:
# Criterio: cantidad de bad events por servicio
bad_por_servicio = (
    df_critico[df_critico['is_bad']]
    .groupby('service_name')
    .size()
    .reset_index(name='bad_events')
    .sort_values('bad_events', ascending=False)
)

print("Bad events por servicio en el momento crítico:")
display(bad_por_servicio)

Bad events por servicio en el momento crítico:


,service_name,bad_events
1,orders-service,72
0,inventory-service,37
2,payment-service,1


### MENSAJES MAS FRECUENTES EN BAD EVENTS

In [26]:
top_mensajes = (
    df_critico[df_critico['is_bad']]['message']
    .value_counts()
    .head(5)
    .to_frame('cantidad')
)

print("Top 5 mensajes en bad events:")
display(top_mensajes)

Top 5 mensajes en bad events:


,cantidad
message,
Order creation failed - inventory lock timeout,72
Database deadlock detected,37
External dependency error,1


### ENDPOINTS MAS COMPROMETIDOS

In [27]:
# Criterio elegido: cantidad de bad events por endpoint
print("Criterio: cantidad de bad events por endpoint")

top_endpoints = (
    df_critico[df_critico['is_bad']]
    .groupby('endpoint')
    .size()
    .reset_index(name='bad_events')
    .sort_values('bad_events', ascending=False)
    .head(5)
)

display(top_endpoints)

Criterio: cantidad de bad events por endpoint


,endpoint,bad_events
3,/orders/cancel,26
4,/orders/create,25
5,/orders/status,21
1,/inv/reserve,18
2,/inv/stock,13


### SEPERAR INCIDENTE Y BASELINE

In [28]:
# Incidente = la ventana crítica
df_incidente = df[df['window'] == ventana_critica].copy()

# Baseline = todo lo que NO es la ventana crítica
df_baseline = df[df['window'] != ventana_critica].copy()

print("Eventos incidente:", len(df_incidente))
print("Eventos baseline:", len(df_baseline))

Eventos incidente: 189
Eventos baseline: 5606
